# DL-thon 위험도 평가 시스템 — Risk Assessment Pipeline

**"한문철 TV + 스카우터" 컨셉**: 오토바이 블랙박스 야간 영상에서 위험도 자동 평가.

**최종 결과**: Domain 10장 GT **Exact 9/9 (100%)** · Test 30장 holdout **자연스러운 분포**

**역할 분담**: 팀 본체 = Segmentation 학습 · 내 파트 = **후처리 Layer** (거리 + 위험도 + HUD)

---
**기간**: 2026-04-22 ~ 2026-04-24 (3일) · **발표**: 2026-04-26


## 1. 프로젝트 진행 중 세운 가설들

시스템 설계의 방향을 결정했던 핵심 가설 6가지. **가설이 맞는지 data 로 검증** 하며 시스템을 발전시킴.

| # | 가설 | 검증 결과 | 설계 반영 |
|---|------|----------|----------|
| **H1** | "정상 운전자는 차선을 지킨다" | ✓ GT 10/10 직진 → 차선 ↔ 정상 주행 1:1 대응 | Lane violation 감지 (strict/x1) |
| **H2** | "거의 모든 프레임이 직진" | ✓ GT 라벨 수집 후 10/10 확인 | `heading_threat` flag 제거 (noise) |
| **H3** | "바이크 블랙박스 = 이미지 중앙 하단 기준" | ✓ 카메라 prior · 물리적 fact | **Ego mask = 중앙 하단 anchor** 로 class 5\|6 선정 |
| **H4** | "점수 합산 자체가 잘못된 추상화" ⭐ | ✓ 터닝포인트 — v3.x 튜닝 지옥 탈출 | **4 독립 경고** (FCW/LDW/BSW/HEAD), `bin = max(severities)` |
| **H5** | "거리 metric 값보다 zone + confidence" | ✓ 3 agent 합의 (Mobileye/OpenPilot/ADAS 업계) | **Zone Corridor** (parametric 4-zone 사다리꼴) |
| **H6** | "모르는 건 모른다" (측정 불가도 valid) | ✓ Test 30 FP 확인 · 실환경 lane 감지 어려움 | **측정 불가 guard** (LDW lateral > 70%) |

**가설 주도 개발**:
* 각 가설이 시스템의 한 축 담당
* "복잡도 추가" 대신 "관점 전환" 으로 돌파구 찾음


## 2. 시스템 아키텍처 — v5 Final

후처리 5-layer 파이프라인. 각 layer 가 특정 가설 구현:

![Architecture](presentation_assets/architecture.png)

**층별 요약**:
1. **Seg Ensemble** — 5-fold 학습 모델 majority vote (Lane Mark IoU +2.2%p)
2. **Ego Anchor (H3)** — 중앙 하단 closest CC 로 my bike mask 선정
3. **VP + Lane + Depth** — 기본 feature 추출
4. **Distance Estimator** — 3-estimator fusion (ground/size/depth)
5. **Zone Corridor (H5)** — Parametric 4-zone (critical/danger/caution/sidelobe)
6. **4 Warning (H4)** — FCW, LDW (측정불가 H6), BSW, HEAD (cut-in vs 역주행 구분)
7. **Bin = max(severities)** — 가중치 합산 X


## 3. 시스템 진화 — v2.6 → v5 final

각 버전별 **Domain 10 프레임 정확도** 변화. **터닝포인트 (H4 - 점수 합산 포기)** 이후 정확도 반등.

![Evolution](presentation_assets/evolution.png)

### 구간별 해석

| 구간 | 버전 | Exact | Lenient | 메시지 |
|---|---|---|---|---|
| **시작** | v2.6 (9-flag 벌점) | 5/10 | 8/10 | baseline — 점수 가산 방식 |
| **실패** | v3.3 (hierarchical) | 3/10 | 7/10 | 계층 구조는 과대평가 유발 |
| **지옥** | v3.3.1 (tweaking) | 2/10 | 6/10 | 파라미터 튜닝은 local optimum |
| **실패** | v4 (distance 기반) | 1/10 | 5/10 | metric 에 너무 의존 |
| **돌파구** | v5 base (4 warnings) | 6/10 | 9/10 | **점수 합산 포기 → 독립 경고** (H4) |
| **개선** | v5 + zone + ensemble | 7-8/9 | 9/9 | Zone clamp + 5-fold ensemble (H5) |
| **최종** | v5 + 측정불가 | **9/9** | **9/9** | **lateral > 70% 불가 처리** (H6) |


## 4. 핵심 실험 기록

### 4.1 Ego Mask — 중앙 하단 anchor (H3)

**문제**: seg 모델이 내 오토바이 앞부분을 class 4 Moveable 로 오분류 → 가짜 근접 경고.

**가설**: 바이크 dashcam 은 헬멧/핸들바 장착 → "나" 는 이미지 중앙 하단.

**해결**:
```python
bike_mask = (mask == 5) | (mask == 6)  # MyBike + Rider
closed = morphology_close(bike_mask)
dilated = dilate(closed, kernel=5)
# 여러 CC 중 (W/2, H) 에 가장 가까운 것만 ego 로 선택
ego_cc = min(CCs, key=lambda cc: dist(cc.centroid, (W/2, H)))
```

**효과**: Frame 1 정면 거리 9.2m → 정확 (이전 4m 는 ego bike 일부)


### 4.2 Zone Corridor (H5) — "가상 삼각형"

**3 agent 리서치** 수렴:
* Bbox foot-point (center 아님) = 노면 접지점
* Lane-width 로 정규화된 signed distance (해상도 불변)
* Clamp / additive layer (대체 X)

**구현**:
```python
# Parametric: y = vp_y + alpha * (H-1-vp_y), 해상도 독립
zones = {
    'critical': alpha=0.85,   # 하위 15% (치명)
    'danger':   alpha=0.55,   # 하위 45% (위험)
    'caution':  alpha=0.15,   # 하위 85% (주의)
    'sidelobe': lane_width × 0.30,  # BSW 용
}
```

**효과**: Frame 6 (bbox 54% FCW-3 오판) → L2 위험 으로 clamp. Frame 11 우측 차도 sidelobe 로 감지.


### 4.3 Depth Pro 실험 (실패 but 교훈)

**시도**: Apple Depth Pro (metric + focal 자동 추정) 로 완전 교체.

**결과**:
- ✓ 기술적으로 동작 (VRAM 1.9GB, f=1022~393 auto)
- ✓ Frame 11 우측 차 실제 3m 로 확인
- ✗ GT 라벨 체계와 distance 기준 불일치 → Exact 9→6 하락

**교훈**: 더 정확한 거리가 반드시 더 좋은 판정은 아님. **GT 체계 = 거리보다 시나리오 기반**.

**롤백**: DA V2 유지.


### 4.4 5-fold Ensemble — 근본 해결

**발견**: 이미 학습된 fold 0~4 모델 5개 존재. **Majority vote** 로 boundary 안정화.

**Test 30 holdout 실측**:

![Ensemble mIoU](presentation_assets/ensemble_miou.png)

**핵심 효과**:
- **Lane Mark IoU +2.2%p** (0.325 → 0.346) — LDW/corridor 정확도 직결
- Frame 11 우측 차 자동 감지 (사용자 지적 해결)
- Ego fragment 필터가 거의 불필요 (filt=0 다수) — mask 자체 깨끗해짐


### 4.5 HEAD cut-in 구분

**문제**: Frame 4 SUV (옆에서 cut-in) 를 "역주행" 으로 오판.

**수정**: HEAD 조건을 "진짜 정면" 으로 제한:
```python
is_frontal_center = (0.40 <= cx_rel <= 0.60)   # 중앙 20%
is_frontal_aspect = (aspect >= 1.8)            # 넓고 낮은 정면 차량
if not (center AND aspect):
    return L0 "cut-in 가능성"   # BSW 로만 감지
```

**효과**: Frame 4 HEAD 해제 → cut-in 으로 정확 분류.


### 4.6 LDW 측정 불가 (H6) — 최종 돌파구

**사용자 통찰**: *"실제 도로에는 차선 인식 어려운 환경이 많다. 너무 쏠리면 인식 불가로 처리하자."*

**구현**:
```python
if lateral_ratio > 0.7:  # lane 폭의 70% 이상 편차 → seg 실패 추정
    return WarningResult('LDW', 0, '측정 불가 (lane 감지 불안정)')
# 이하는 기존 편향 L1 / 이탈 L2 로직 유지
```

**효과**:
- Frame 1, 5, 11: 극단적 lateral (76~81%) → 측정 불가 처리
- **Exact 9/9 달성** (Frame 1 위험 FP 제거, Frame 5 이전 MISS 였던 것도 복귀)
- Test 30 LDW FP 53% → 33%


## 5. 최종 메트릭

### 5.1 Domain 10 프레임 상세 (사용자 GT)


In [ ]:
import json, sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
from pathlib import Path

ARCHIVE = Path(r'C:\Users\akals\Downloads\archive')

with open(ARCHIVE / 'v5_results.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

# 정확도 계산
exact = lenient = total = 0
rows = []
for k, v in results.items():
    gt = v['gt']
    if gt is None:
        rows.append((k[-10:-4], v['bin'], '?', '-', v['active_warnings']))
        continue
    total += 1
    is_exact = v['bin'] == gt
    is_len = is_exact or \
             (gt == '주의' and v['bin'] in ['주의', '위험']) or \
             (gt == '위험' and v['bin'] in ['주의', '위험', '치명'])
    if is_exact: exact += 1
    if is_len: lenient += 1
    status = 'EXACT' if is_exact else ('OK' if is_len else 'MISS')
    rows.append((k[-10:-4], v['bin'], gt, status, v['active_warnings']))

print(f'▶ Exact: {exact}/{total} ({exact/total*100:.0f}%)')
print(f'▶ Lenient: {lenient}/{total} ({lenient/total*100:.0f}%)')
print()
print(f'{"Frame":<8}{"pred":<6}{"GT":<6}{"status":<8}{"active warnings"}')
print('-' * 60)
for nm, pred, gt, status, warns in rows:
    print(f'{nm:<8}{pred:<6}{gt:<6}{status:<8}{",".join(warns) or "-"}')


### 5.2 Domain vs Test 30 — 상황 구분 능력

![Distribution](presentation_assets/distribution_comparison.png)

**시스템이 정확히 다른 상황을 반영**:
- Domain (사고 집중): 치명 30%, 위험 22%
- Test 30 (일반 주행): 치명 0%, 안전 63% — **자연스러움**

이는 **overfitting 아님** 을 시사 (10 프레임만 외운 게 아님).


### 5.3 Test 30 holdout — 경고 활성 빈도 (FP check)

| 경고 | 이전 (편향 L1) | 최종 (측정불가 guard) |
|---|---|---|
| FCW | 0% | 0% (정면 거리 중앙값 45m, 멀어서 자연스러움) |
| **LDW** | **53%** (FP 많음) | **33%** ✓ |
| BSW | 7% | 7% |
| HEAD | 0% | 0% |

**LDW 53% → 33%**: 측정 불가 guard 로 seg 노이즈 FP 제거. 실제 편향/이탈만 남음.


## 6. 핵심 설계 철학

### 6.1 "점수 합산" 에서 "독립 경고" 로 (H4)

**Before (v2.6)**:
```
score = sum(flag_weight)   # 이질적 신호 덧셈, 자의적 가중치
bin = threshold(score)     # threshold 튜닝 지옥
```

**After (v5)**:
```
FCW, LDW, BSW, HEAD = independent_severity    # 각 축 독립
bin = max(severities)                          # 최고 심각도
reason = active_warnings                       # 해석 가능
```

**이득**:
- 가중치 자의성 제거
- "왜 치명?" 답 명확 ("FCW-3 + BSW-2 + HEAD-3")
- 튜닝 포인트 감소 (12 → 9 threshold)

### 6.2 Domain Gap 을 후처리로 — 한계 인정

**현실**: 학습 데이터 (200장) 와 실제 도메인 분포 다름.

**접근**:
- Seg 재학습 대신 **후처리 5-layer** 로 노이즈 보정
- Ensemble 로 Lane Mark +2.2%p 향상
- 근본 해결은 **더 강한 pretrained + domain 특화 데이터** (future work)

### 6.3 "모르는 건 모른다" (H6)

**실제 도로 환경의 다양성**:
- 야간 저조도 → lane mark 안 보임
- 공사 구역, 교차로, 터널 → 임시/비정형 marking
- 비/눈 → boundary 흔들림

**설계 원칙**:
- 애매하면 침묵 (측정 불가)
- 잘못된 경고 > no-signal (실전에서 위험)
- Confidence threshold + safeguard 다수 적용

**구현 위치**:
- LDW: `lateral > 70%` → 측정 불가
- HEAD: cut-in 가능성 → L0
- Distance: estimator 실패 시 None
- Zone: VP conf < 0.3 → corridor disable
- Ego mask: class 5 픽셀 < 500 → filter 비활성


## 7. HUD 시각화 — Domain 10 프레임

각 프레임별 HUD: 경고 bbox 색상 구분, 거리 표시, zone overlay, LDW 편향/측정불가 표시.

![Domain Gallery](v5_gallery.png)

**예시 Frame (발표 요약)**:

### Frame 5 (120122 SUV) — 치명
* SUV cut-in: FCW + LDW 편향 + BSW 좌측근접 + (HEAD → cut-in 으로 제외)
* **3 독립 경고 동시 → 치명**

### Frame 10 (144602 역주행 추월) — 치명
* 정면 트럭 2.1m + 우측 차량 2.1m + HEAD (중앙 정면 + aspect 넓음)
* **진짜 역주행 정확 감지**

### Frame 11 (212120) — 사용자 지적 해결
* 이전: BSW L0 (10.6m 과대 추정)
* **Ensemble 적용 → R=4.7m → BSW L1 정확 감지**

### Frame 1 (114941) — 측정 불가 처리
* 이전: LDW 편향 76% 로 위험 오판
* **측정 불가 guard → FCW-1 주의만 남음 (GT 주의 EXACT)**


## 8. Test 30장 holdout 샘플
일반 주행 분포 (사고 장면 없음) — 시스템이 **"정상 주행"** 도 정확히 식별 (치명 0%).

![Test Gallery](v5_testset_gallery.png)


## 9. Limitation & Future Work

### 9.1 현재 한계

| 영역 | 한계 | 완화 현황 |
|---|---|---|
| **Seg boundary** | class 4/5 (Moveable vs MyBike) confusion | Ensemble + ego mask |
| **Lane Mark** | IoU 0.346 (상대적으로 낮음) | 측정 불가 guard 로 FP 제어 |
| **야간 depth** | DA V2 가 near-field 일부 오분류 | Depth weight 0.05 로 낮춤 |
| **곡선 도로** | VP 가정 무너짐 | corridor disable safeguard |
| **모터사이클 lean** | 카메라 tilt → corridor 기울어짐 | 현재 데이터 직진 위주라 미발현 |
| **Multi-frame** | TTC / optical flow 미활용 | Single-frame 한계 인정 |

### 9.2 Future Work

1. **Segmentation 재학습** — 더 강한 pretrained encoder (Segformer MiT-B5 / Swin-L)
2. **도메인 특화 데이터 augmentation** — 야간/저조도 특화
3. **Multi-frame TTC** — bbox size 변화율로 충돌 시간 예측
4. **Lane Marking 독립 학습** — Mapillary Vistas 등 lane mark 포함 dataset
5. **Domain adaptation** — 소수 pseudo-label 기반 fine-tune

### 9.3 발표 3대 메시지

1. **도메인 갭**을 seg 재학습 없이 **후처리 5-layer** 로 극복 → **Exact 9/9**
2. **경고 기반 판정** (점수 합산 X) 으로 **해석 가능성** 확보
3. **자기 한계 인식** (측정 불가 명시) 으로 **실전 robustness** 보증


---
### 생성 파일
* `_v5_warnings.py` — 메인 시스템 (4 warnings + zone + ensemble + 측정 불가)
* `ego_corridor.py` — 4-zone parametric corridor
* `geometric_distance.py` — 3-estimator fusion
* `v5_results.json` — Domain 10 결과 (Exact 9/9)
* `v5_testset_results.json` — Test 30 결과
* `v5_gallery.png` — Domain HUD 갤러리
* `v5_testset_gallery.png` — Test 30 HUD 갤러리
* `presentation_assets/` — 발표용 도표 (architecture, evolution, ensemble, distribution)

*문서 생성: 2026-04-24 — DL-thon Day 3*
